# QuetHoaDon · Đánh giá cuối (Tuần 5): cấu hình đóng băng, demo ảnh → JSON

Chạy trên Kaggle: **GPU T4**, Internet bật. Gắn 4 dataset:
`m1b-v2-rs` (checkpoint M1b Tuần 4), `quethoadon` (token dataset v1), **Vietnamese Receipts MC_OCR 2021** (ảnh gốc + CSV),
`phase1-result` (`m1b_v3_checkpoint` + `phase1_eval`: VietOCR tinh chỉnh, token dataset `tok_ft_n05`, báo cáo validation).

**Cấu hình cuối (đã đóng băng):** tiền xử lý → CRAFT → VietOCR tinh chỉnh vòng 1 → lọc hộp chồng nhau 0,5 → M1b v3
(run `m1b_v3_ft_n05_20261117_010041`) → luật ngày giờ → làm sạch SELLER / ADDRESS → đọc mặt hàng bằng luật.

Notebook này **không chọn gì**: nó chạy lại cấu hình đã đóng băng và kiểm tra ra đúng các số đã báo cáo. Tập test đã được
chấm một lần ngày 17/11/2026; chạy lại ở đây là tính lại cùng một kết quả để có output lưu trong notebook, không phải lần
chấm mới để chọn cấu hình. Mỗi cell `[KIỂM THỬ]` in **KHỚP** hoặc **LỆCH**.

Cách chạy khuyên dùng: **Save Version → Save & Run All (Commit)**. Kaggle chạy nền (máy bạn ngủ cũng không sao) và lưu
notebook kèm output. Tổng thời gian khoảng 30–40 phút, phần lớn là cài detectron2 và demo OCR.

In [1]:
# CELL 1 - Cài detectron2 (LayoutXLM cần, 5-10 phút). Mong đợi: "detectron2 OK"
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

detectron2 OK 0.6


In [2]:
# CELL 2 - Lấy mã nguồn đúng commit đã chạy notebook này (main đã đổi luật mặt hàng sau đó) và cài thư viện OCR. Mong đợi: một dòng commit 413599d, "xong"
COMMIT = "413599d"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f {COMMIT} && git log --oneline -1
!pip install -q -r requirements-dev.txt
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb 2>&1 | tail -2
print("xong")

/kaggle/working
/kaggle/working/QuetHoaDon
413599d (HEAD -> main, origin/main, origin/HEAD) Add the final evaluation notebook, the Week 5 final report and bring the docs up to date (#17)
vietocr 0.3.13 requires pillow==10.2.0, but you have pillow 13.0.0 which is incompatible.
vietocr 0.3.13 requires prefetch-generator==1.0.1, but you have prefetch-generator 1.0.3 which is incompatible.
xong


In [3]:
# CELL 3 [KIỂM THỬ] - GPU, dữ liệu, hai checkpoint, trọng số VietOCR và unit test.
# Mong đợi: "GPU: True Tesla T4", mọi đường dẫn có giá trị, "Ran ... tests" rồi "OK"
import glob, os, sys, json, csv, torch
sys.path.insert(0, '/kaggle/working/QuetHoaDon')
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
one = lambda pattern: sorted(glob.glob(pattern, recursive=True))[-1]
TOK = os.path.dirname(one('/kaggle/input/**/token_dataset_v1*/**/validation.jsonl'))
CSV = one('/kaggle/input/**/mcocr_train_df.csv')
IMG_RAW = os.path.dirname(one('/kaggle/input/**/train_images/**/*.jpg'))
CKPT = one('/kaggle/input/**/runs/m1b_v2_*/best')
TOKC = os.path.dirname(one('/kaggle/input/**/phase1_eval/tok_ft_n05/validation.jsonl'))
CK3 = os.path.dirname(one('/kaggle/input/**/m1b_v3_checkpoint/best/config.json'))
VFT1 = one('/kaggle/input/**/phase1_eval/vietocr_ft/vietocr_finetuned.pth')
REF = one('/kaggle/input/**/phase1_eval/eval/m1b_v2_val_now.json')
for k in ("TOK", "CSV", "IMG_RAW", "CKPT", "TOKC", "CK3", "VFT1", "REF"):
    print(f"{k} = {globals()[k]}")
assert os.path.basename(os.path.dirname(CKPT)) == "m1b_v2_20261110_183118", "checkpoint Tuần 4 sai"
W = '/kaggle/working'
os.makedirs(f'{W}/eval', exist_ok=True)
!python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"

GPU: True Tesla T4
TOK = /kaggle/input/datasets/bluegogeta/quethoadon/token_dataset_v1-20261110T164902Z-1-001/token_dataset_v1
CSV = /kaggle/input/datasets/domixi1989/vietnamese-receipts-mc-ocr-2021/mcocr_train_df.csv
IMG_RAW = /kaggle/input/datasets/domixi1989/vietnamese-receipts-mc-ocr-2021/train_images/train_images
CKPT = /kaggle/input/datasets/bluegogeta/m1b-v2-rs/runs/m1b_v2_20261110_183118/best
TOKC = /kaggle/input/datasets/bluegogeta/phase1-result/phase1_eval/tok_ft_n05
CK3 = /kaggle/input/datasets/bluegogeta/phase1-result/m1b_v3_checkpoint/best
VFT1 = /kaggle/input/datasets/bluegogeta/phase1-result/phase1_eval/vietocr_ft/vietocr_finetuned.pth
REF = /kaggle/input/datasets/bluegogeta/phase1-result/phase1_eval/eval/m1b_v2_val_now.json
Ran 279 tests in 2.169s
OK


In [4]:
# CELL 4 [KIỂM THỬ] - File chia tập và ảnh tiền xử lý của validation + test (CPU, vài phút, có thanh tiến trình).
# Mong đợi: "validation: 115 hóa đơn, thiếu ảnh 0" và "test: 116 hóa đơn, thiếu ảnh 0"
import cv2
from tqdm.auto import tqdm
from src.common.config import load_config
from src.preprocessing.image_preprocessor import ImagePreprocessor
from src.training.data import load_jsonl
!python scripts/prepare_dataset.py --csv "{CSV}" --images "{IMG_RAW}" --out {W}/dataset > {W}/dataset_summary.txt
MANIFEST = f'{W}/dataset/mcocr_manifest_split.csv'
with open(MANIFEST, encoding='utf-8') as f:
    split_rows = list(csv.DictReader(f))
IMG_PRE = f'{W}/img_pre'
os.makedirs(IMG_PRE, exist_ok=True)
pp = ImagePreprocessor(**load_config('/kaggle/working/QuetHoaDon/configs')['model']['preprocessing'])
for r in tqdm([r for r in split_rows if r['split'] in ('validation', 'test')], desc="tiền xử lý"):
    out = os.path.join(IMG_PRE, os.path.splitext(r['img_id'])[0] + '.jpg')
    if not os.path.exists(out):
        cv2.imwrite(out, pp.process(os.path.join(IMG_RAW, r['img_id'])).image, [cv2.IMWRITE_JPEG_QUALITY, 95])
for sp in ('validation', 'test'):
    recs = load_jsonl(os.path.join(TOKC, f'{sp}.jsonl'))
    miss = [r['image'] for r in recs if not os.path.exists(os.path.join(IMG_PRE, r['image']))]
    print(f"{sp}: {len(recs)} hóa đơn, thiếu ảnh {len(miss)}")

tiền xử lý:   0%|          | 0/1307 [00:00<?, ?it/s]

validation: 115 hóa đơn, thiếu ảnh 0
test: 116 hóa đơn, thiếu ảnh 0


In [5]:
# CELL 5 [KIỂM THỬ] - Tái lập kết quả validation của cấu hình cuối (GPU, vài phút).
# Mong đợi: bảng "Strict reading accuracy" và dòng "validation ... KHỚP"
def strict_counts(path):
    a = json.load(open(path))['app']
    f = a['fields']
    return {"SELLER": f['SELLER']['correct'], "ADDRESS": f['ADDRESS']['correct'], "date": f['TIMESTAMP']['correct'],
            "time": f['TIMESTAMP']['time']['correct'], "TOTAL": f['TOTAL_COST']['correct'],
            "all": a['receipts_all_correct']['correct']}

def check(name, got, expected):
    print(f"{name}: {got} -> {'KHỚP' if got == expected else 'LỆCH, mong đợi ' + str(expected)}")

!python scripts/evaluate_layoutxlm.py --checkpoint "{CK3}" --token-dir {TOKC} --image-dir {IMG_PRE} --split validation --save-probs {W}/eval/m1b_v3_val_probs.npz --out {W}/eval/m1b_v3_val.json 2>&1 | tail -3
!python scripts/decode_from_probs.py --token-dir {TOKC} --split validation --probs {W}/eval/m1b_v3_val_probs.npz --datetime --cleanup --out {W}/eval/decoded_v3_val.json | grep -A10 "Strict reading"
check("validation, cấu hình cuối", strict_counts(f'{W}/eval/decoded_v3_val.json'),
      {"SELLER": 74, "ADDRESS": 29, "date": 79, "time": 54, "TOTAL": 75, "all": 20})

token ROC-AUC macro: 0.8378513532621447
entity micro F1: 0.4183
full report: /kaggle/working/eval/m1b_v3_val.json
### Strict reading accuracy
| Field | Correct | Accuracy | 95 % interval | No prediction | Mean CER |
|---|---|---|---|---|---|
| SELLER | 74 / 106 | 69.8 % | [60.5, 77.7] | 4 | 0.231 |
| ADDRESS | 29 / 112 | 25.9 % | [18.7, 34.7] | 4 | 0.316 |
| TIMESTAMP | 79 / 99 | 79.8 % | [70.8, 86.5] | 16 | - |
| TOTAL_COST | 75 / 98 | 76.5 % | [67.2, 83.8] | 3 | - |
| TIMESTAMP time hh:mm | 54 / 80 | 67.5 % | [56.6, 76.8] | - | - |
| **All fields correct** | 20 / 115 | 17.4 % | [11.5, 25.3] | - | - |

error causes: {'annotation_without_amount': 8, 'no_ocr_word_labelled': 0, 'ocr_text_error': 71, 'model_missed': 2, 'model_wrong': 32}
validation, cấu hình cuối: {'SELLER': 74, 'ADDRESS': 29, 'date': 79, 'time': 54, 'TOTAL': 75, 'all': 20} -> KHỚP


In [6]:
# CELL 6 [KIỂM THỬ] - Tính lại kết quả test của cấu hình đóng băng và của mốc Tuần 4, rồi so sánh ghép cặp (GPU, vài phút).
# Đây là tính lại đúng kết quả đã chấm ngày 17/11/2026, không chọn gì. Mong đợi: hai dòng "KHỚP" và bảng ghép cặp
!python scripts/evaluate_layoutxlm.py --checkpoint "{CK3}" --token-dir {TOKC} --image-dir {IMG_PRE} --split test --allow-test --save-probs {W}/eval/m1b_v3_test_probs.npz --out {W}/eval/m1b_v3_test.json 2>&1 | tail -2
!python scripts/decode_from_probs.py --token-dir {TOKC} --split test --allow-test --probs {W}/eval/m1b_v3_test_probs.npz --datetime --cleanup --out {W}/eval/decoded_v3_test.json | grep -A10 "Strict reading"
!python scripts/evaluate_layoutxlm.py --checkpoint "{CKPT}" --token-dir {TOK} --image-dir {IMG_PRE} --split test --allow-test --save-probs {W}/eval/m1b_v2_test_probs.npz --out {W}/eval/m1b_v2_test.json 2>&1 | tail -2
!python scripts/decode_from_probs.py --token-dir {TOK} --split test --allow-test --probs {W}/eval/m1b_v2_test_probs.npz --datetime --out {W}/eval/decoded_v2_test.json | grep -A10 "Strict reading"
check("test, cấu hình cuối", strict_counts(f'{W}/eval/decoded_v3_test.json'),
      {"SELLER": 77, "ADDRESS": 29, "date": 89, "time": 61, "TOTAL": 71, "all": 21})
check("test, Tuần 4 + luật ngày giờ", strict_counts(f'{W}/eval/decoded_v2_test.json'),
      {"SELLER": 61, "ADDRESS": 17, "date": 83, "time": 43, "TOTAL": 64, "all": 11})
!python scripts/compare_strict.py --token-dir {TOKC} --split test --allow-test --a week4={W}/eval/decoded_v2_test.json --b final={W}/eval/decoded_v3_test.json --out {W}/eval/paired_test.json

entity micro F1: 0.4267
full report: /kaggle/working/eval/m1b_v3_test.json
### Strict reading accuracy
| Field | Correct | Accuracy | 95 % interval | No prediction | Mean CER |
|---|---|---|---|---|---|
| SELLER | 77 / 107 | 72.0 % | [62.8, 79.6] | 2 | 0.156 |
| ADDRESS | 29 / 106 | 27.4 % | [19.8, 36.5] | 2 | 0.223 |
| TIMESTAMP | 89 / 106 | 84.0 % | [75.8, 89.7] | 10 | - |
| TOTAL_COST | 71 / 98 | 72.4 % | [62.9, 80.3] | 5 | - |
| TIMESTAMP time hh:mm | 61 / 85 | 71.8 % | [61.4, 80.2] | - | - |
| **All fields correct** | 21 / 114 | 18.4 % | [12.4, 26.5] | - | - |

error causes: {'annotation_without_amount': 10, 'no_ocr_word_labelled': 2, 'ocr_text_error': 52, 'model_missed': 5, 'model_wrong': 31}
entity micro F1: 0.4038
full report: /kaggle/working/eval/m1b_v2_test.json
### Strict reading accuracy
| Field | Correct | Accuracy | 95 % interval | No prediction | Mean CER |
|---|---|---|---|---|---|
| SELLER | 61 / 107 | 57.0 % | [47.5, 66.0] | 1 | 0.244 |
| ADDRESS | 17 / 106 | 16.0 % |

In [7]:
# CELL 7 [KIỂM THỬ] - Đọc mặt hàng bằng luật, chấm bằng tổng tiền chuẩn (CPU, vài giây).
# Mong đợi test: "item sum = annotated total | 46 / 97" và dòng "KHỚP"
!python scripts/score_items.py --token-dir {TOKC} --split validation --out {W}/eval/items_val.json | sed -n 1,12p
!python scripts/score_items.py --token-dir {TOKC} --split test --allow-test --out {W}/eval/items_test.json | sed -n 1,12p
it = json.load(open(f'{W}/eval/items_test.json'))
check("mặt hàng, test", (it['sum_matches_gold_total']['count'], it['receipts_with_gold_total']), (46, 97))

### Item reading, validation (94 receipts with an annotated total)
| Check | Count | Rate | 95 % interval |
|---|---|---|---|
| at least one item read | 80 / 94 | 85.1 % | [76.5, 90.9] |
| item sum = annotated total | 59 / 94 | 62.8 % | [52.7, 71.9] |
| two or more items read | 50 / 94 | 53.2 % | [43.2, 63.0] |
| of those, item sum = annotated total | 36 / 50 | 72.0 % | [58.3, 82.5] |
| verified (sum = printed total line) | 52 / 94 | 55.3 % | [45.3, 65.0] |
| of verified, sum = annotated total | 51 / 52 | 98.1 % | [89.9, 99.7] |
| column header found | 76 / 94 | 80.9 % | [71.7, 87.5] |
| total line found | 92 / 94 | 97.9 % | [92.6, 99.4] |
| item lines with qty x price = amount | 187 / 208 | 89.9 % | [85.1, 93.3] |
### Item reading, test (97 receipts with an annotated total)
| Check | Count | Rate | 95 % interval |
|---|---|---|---|
| at least one item read | 78 / 97 | 80.4 % | [71.4, 87.1] |
| item sum = annotated total | 46 / 97 | 47.4 % | [37.8, 57.3] |
| two or more items read | 58

In [8]:
# CELL 8 [KIỂM THỬ] - Đường đi của app (từng hóa đơn một) có tái lập dự đoán đã chấm không (GPU, vài phút).
# Mong đợi: "identical 115" và "PASS"
!python scripts/verify_extractor.py --checkpoint "{CKPT}" --token-dir {TOK} --image-dir {IMG_PRE} --reference {REF} --out {W}/eval/verify_extractor_val.json 2>&1 | tail -6

receipts compared 115 / 115, identical 115
field mismatches: {'SELLER': 0, 'ADDRESS': 0, 'TIMESTAMP': 0, 'TOTAL_COST': 0}
scores reference: {'SELLER': 0.6884, 'ADDRESS': 0.5, 'TIMESTAMP': 0.7594, 'TOTAL_COST': 0.689, 'macro_f1': 0.6592, 'receipts_all_correct': 34}
scores now      : {'SELLER': 0.6884, 'ADDRESS': 0.5, 'TIMESTAMP': 0.7594, 'TOTAL_COST': 0.689, 'macro_f1': 0.6592, 'receipts_all_correct': 34}
PASS
written: /kaggle/working/eval/verify_extractor_val.json


In [9]:
# CELL 9 [KIỂM THỬ] - Độ trễ: OCR với VietOCR tinh chỉnh trên ảnh test, rồi khâu LayoutXLM trên GPU và CPU (khoảng 8 phút).
# Mong đợi: GPU vài chục ms mỗi hóa đơn, CPU khoảng 1 giây; OCR khoảng 2,5 giây mỗi hóa đơn
OCR_T = f'{W}/ocr_test_timing'
!python -u scripts/run_ocr_dataset.py --images "{IMG_RAW}" --split-manifest {MANIFEST} --splits test --out {OCR_T} --weights {VFT1} --nested-threshold 0.5 2>&1 | grep -E "^\[(ocr|done)\]"
!python scripts/measure_latency.py --token-dir {TOKC} --image-dir {IMG_PRE} --split test --checkpoint M1b_v3={CK3} --ocr-dir {OCR_T}/ocr_annotations --devices cuda,cpu --n-cpu 10 --out {W}/eval/latency_final.json 2>&1 | grep -v "Loading weights"

[ocr] 116 images (missing on disk: 0) | device=cuda | weights=/kaggle/input/datasets/bluegogeta/phase1-result/phase1_eval/vietocr_ft/vietocr_finetuned.pth | nested_threshold=0.5
[done] {'ok': 116, 'empty': 0, 'skipped': 0, 'failed': 0} | 268s | OCR crop errors: 0
[M1b_v3/cuda] 58.9 ms mean over 60 receipts
[M1b_v3/cpu] 913.2 ms mean over 10 receipts
| System | Device | Receipts | Mean ms | Median ms | p95 ms | Parameters | Size MB |
|---|---|---|---|---|---|---|---|
| M1b_v3 | cuda | 60 | 58.9 | 54.8 | 86.0 | 368,658,377 | 1476 |
| M1b_v3 | cpu | 10 | 913.2 | 908.2 | 951.1 | 368,658,377 | 1476 |

OCR stages (mean seconds per receipt): {'preprocess': 0.106, 'detect': 0.606, 'ocr': 1.718, 'features': 0.001, 'total': 2.431}
written: /kaggle/working/eval/latency_final.json


In [10]:
# CELL 10 - DEMO Data → Model → Output: ảnh hóa đơn gốc → JSON gọn (4 trường + mặt hàng), 3 hóa đơn test (GPU, vài phút).
# Trường nào không đọc được thì không có trong JSON; "items_check.status" = "verified" khi tổng các món bằng tổng in trên hóa đơn.
from src.pipeline.extractor import ReceiptExtractor, compact_result
ex = ReceiptExtractor.from_checkpoint(CK3, '/kaggle/working/QuetHoaDon/configs', 'cuda', with_ocr=True, postprocess=True,
                                      ocr_weights=VFT1, items=True, nested_threshold=0.5)
os.makedirs(f'{W}/demo', exist_ok=True)
demo_ids = [r['img_id'] for r in split_rows if r['split'] == 'test'][:3]
for img_id in demo_ids:
    res = ex.extract(os.path.join(IMG_RAW, img_id))
    out = compact_result(res)
    json.dump(out, open(f"{W}/demo/{os.path.splitext(img_id)[0]}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("=====", img_id, "| thời gian (giây):", res['timing_sec'])
    print(json.dumps(out, ensure_ascii=False, indent=2))

Loading weights:   0%|          | 0/746 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(


===== mcocr_public_145013dhzmu.jpg | thời gian (giây): {'preprocess': 0.082, 'detect': 1.065, 'ocr': 1.269, 'features': 0.001, 'total': 2.866, 'layoutxlm': 0.45}
{
  "file_id": "mcocr_public_145013dhzmu",
  "fields": {
    "TOTAL_COST": {
      "text": "418,986",
      "amount_vnd": 418986
    }
  }
}
===== mcocr_public_145013tumhj.jpg | thời gian (giây): {'preprocess': 0.123, 'detect': 0.47, 'ocr': 1.195, 'features': 0.001, 'total': 1.845, 'layoutxlm': 0.057}
{
  "file_id": "mcocr_public_145013tumhj",
  "fields": {
    "TIMESTAMP": {
      "text": "15/08/2020",
      "date": "15/08/2020"
    },
    "TOTAL_COST": {
      "text": "585,500",
      "amount_vnd": 585500
    },
    "SELLER": {
      "text": "Phúc Anh Minimart"
    },
    "ADDRESS": {
      "text": "248 Cổ Bi - Gia Lâm Hà Nội"
    }
  },
  "items": [
    {
      "name": "TH Ít Đường 110m",
      "quantity": 1,
      "unit_price": 238000,
      "amount": 238000
    },
    {
      "name": "Tã Quần Huggie",
      "quantity": 1,

In [11]:
# CELL 11 - RQ1: tiền xử lý ảnh ảnh hưởng thế nào tới lỗi ký tự của VietOCR (validation, vùng có nhãn; thời gian chưa đo,
# ước tính 10-20 phút, có thanh tiến trình). Bốn cấu hình: chỉ đổi cỡ, chỉnh nghiêng, tăng tương phản, cả hai; thêm ảnh làm mờ / tối / nhiễu.
# Đo với VietOCR tinh chỉnh (bộ đọc của cấu hình cuối). Mong đợi: bảng CER / WER theo cấu hình. Không chọn gì ở đây.
!python scripts/ocr_preprocessing_ablation.py --csv "{CSV}" --image-dir "{IMG_RAW}" --split-manifest {MANIFEST} --weights {VFT1} --degradations none,blur,dark,noise --out {W}/eval/ocr_ablation_ft.json --md {W}/eval/ocr_ablation_ft.md

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
[*] 115 validation receipts, degradations=['none', 'blur', 'dark', 'noise'], device=cuda
read noise/full: 100%|██████████████████████████| 11/11 [00:45<00:00,  4.18s/it]
| Degradation / config | Regions | CER % | WER % | CER low % | CER mid % | CER high % |
|---|---|---|---|---|---|---|
| blur/contrast | 668 | 44.0 | 72.8 | 56.4 | 39.5 | 37.0 |
| blur/deskew | 668 | 42.0 | 67.1 | 53.2 | 37.1 | 36.4 |
| blur/full | 668 | 43.7 | 72.4 | 54.1 | 38.9 | 38.8 |
| blur/none | 668 | 43.3 | 69.4 | 53.3 | 39.5 | 37.9 |
| dark/contrast | 668 | 10.0 | 24.9 | 13.4 | 8.4 | 8.6 |
| dark/deskew | 668 | 10.3 | 25.0 | 11.8 | 10.0 | 9.2 |
| dark/full | 668 | 10.5 | 25.5 | 11.5 | 11.1 | 9.0 |
| dark/none | 668 |

In [12]:
# CELL 12 - Nén kết quả để tải về (tab Output): eval và demo
!cd /kaggle/working && zip -q -r final_eval.zip eval demo && ls -la final_eval.zip

-rw-r--r-- 1 root root 1855302 Nov 17 18:29 final_eval.zip
